# 19. 전문 검색 전환 후 run() 회귀 확인 (조사 처리 + 샘플 k=20)

- 변경: 검색을 `chunk_text` 전문 인덱스로 바꿨다. 조사는 짧은 목록으로만 뗀다. 샘플 경로는 k=20, 새 카드 경로는 k=10.
- 정책: `PLTest` 라벨과 `test_` 접두사 노드만 넣고, 끝나면 같은 조건으로 지운다.
- 기대: 샘플 질문은 인용 4개(홍역 93쪽 b01, 법 18쪽 b01·b02, 19쪽 b01), Q05는 ANSWERED(93쪽 b04), 무관 질문과 전파 질문은 INSUFFICIENT_EVIDENCE.

In [1]:
import sys
sys.path.insert(0, "../../src")
from rag.prevention_legal.retrieve import load_test_blocks, delete_test_blocks, lucene_query
from rag.prevention_legal.run import run, SAMPLE_QUESTION

DOCS_ALLOWED = ["kdca_measles_2026", "kdca_diagnosis_reporting_2026",
                "moleg_infectious_disease_act_21521", "kdca_hfmd_2026"]
CASES = {
    "샘플": SAMPLE_QUESTION,
    "Q05": "교내 MMR 2차 접종 미완료자는 어떻게 해야 하나요",
    "무관(부작용)": "MMR 2차 접종의 부작용은?",
    "전파": "홍역은 어떻게 전파되나요?",
}
print(lucene_query(CASES["전파"]))

"홍역" OR "어떻게" OR "전파"


In [2]:
# 이번 실행에서 만든 노드 ID 목록. 적재가 실패하면 빈 목록이라 삭제할 것도 없다.
created = []
try:
    created = load_test_blocks(set(DOCS_ALLOWED))
    print("넣은 개수:", len(created))
    for name, q in CASES.items():
        out = run({"question": q, "region": "NATIONAL", "allowed_document_ids": DOCS_ALLOWED})
        print("[" + name + "]", out["status"], "| 인용", len(out["citations"]))
        for c in out["citations"]:
            print("   ", c["chunk_id"])
finally:
    print("삭제한 개수:", delete_test_blocks(created))


넣은 개수: 33


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.fulltext.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('chunk_text', $q) YIELD node, score\n                    WHERE node:PLTest AND node.document_id IN $docs\n                    RETURN node.chunk_id AS chunk_id, node.text AS text, node.document_id AS document_id,\n                           node.pdf_page AS pdf_page, score\n                    ORDER BY scor

[샘플] PARTIAL | 인용 4
    kdca_measles_2026:p0093:b01
    moleg_infectious_disease_act_21521:p0018:b01
    moleg_infectious_disease_act_21521:p0018:b02
    moleg_infectious_disease_act_21521:p0019:b01


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.fulltext.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('chunk_text', $q) YIELD node, score\n                    WHERE node:PLTest AND node.document_id IN $docs\n                    RETURN node.chunk_id AS chunk_id, node.text AS text, node.document_id AS document_id,\n                           node.pdf_page AS pdf_page, score\n                    ORDER BY scor

[Q05] ANSWERED | 인용 1
    kdca_measles_2026:p0093:b04


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.fulltext.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('chunk_text', $q) YIELD node, score\n                    WHERE node:PLTest AND node.document_id IN $docs\n                    RETURN node.chunk_id AS chunk_id, node.text AS text, node.document_id AS document_id,\n                           node.pdf_page AS pdf_page, score\n                    ORDER BY scor

[무관(부작용)] INSUFFICIENT_EVIDENCE | 인용 0


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.fulltext.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('chunk_text', $q) YIELD node, score\n                    WHERE node:PLTest AND node.document_id IN $docs\n                    RETURN node.chunk_id AS chunk_id, node.text AS text, node.document_id AS document_id,\n                           node.pdf_page AS pdf_page, score\n                    ORDER BY scor

[전파] INSUFFICIENT_EVIDENCE | 인용 0


삭제한 개수: 33


In [3]:
from rag.prevention_legal.neo4j_conn import connect
d = connect()
with d.session() as s:
    print("PLTest:", s.run("MATCH (c:PLTest) RETURN count(c) AS n").single()["n"])
    print("test_ ID:", s.run("MATCH (c:Chunk) WHERE c.chunk_id STARTS WITH 'test_' RETURN count(c) AS n").single()["n"])
    print("전체 Chunk:", s.run("MATCH (c:Chunk) RETURN count(c) AS n").single()["n"])
d.close()

PLTest: 0
test_ ID: 0


전체 Chunk: 1569
